# Public Dialogue Analyser

## Jessica K. Montgomery

## May 12th 2026

## Purpose

This notebook analyses 65 UK public dialogue documents on science and technology to identify:

- **Shared concerns or hopes** that recur across different technologies
- **Concerns or benefits that are distinctive to public dialogues on artificial intelligence (AI)** compared to other technologies
- **How public dialogue about AI changes over time**


## Corpus

The corpus consists of UK public dialogue reports on science and technology, spanning multiple technologies (including AI) and multiple years. Each document is associated with metadata including technology domain(s) and publication year.

PDFs are ingested directly and converted to paragraph-level text for analysis.



In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

In [ ]:
import os
import json
import time
import re
from collections import Counter
from datetime import datetime
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import entropy
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from openai import OpenAI
from IPython.display import display, HTML


In [ ]:
# @title Import libraries and define configuration

import random
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150


In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths — resolve sensibly for both Colab (/content/...) and local (repo root)
try:
    import google.colab  # noqa: F401
    _REPO_ROOT = Path("/content")
except ImportError:
    # Local: notebook lives in repo root, so relative paths work directly
    _REPO_ROOT = Path(".")

PDF_FOLDER        = _REPO_ROOT / "pdfs"
OUTPUT_FOLDER     = _REPO_ROOT / "outputs"
CHECKPOINT_FOLDER = _REPO_ROOT / "checkpoints"
DATA_FOLDER       = _REPO_ROOT / "data"

# Ensure output directories exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)


import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

In [ ]:
# @title Load dialogue_utils — shared helper functions
# All helper functions (show_status, show_complete, show_warning,
# save_checkpoint, load_checkpoint, extract_chunks_from_pdf, extract_phrases,
# label_cluster, normalized_entropy, run_sensitivity, etc.) are imported from
# dialogue_utils.py.  The module must be on sys.path (it is fetched in the
# cell below).

try:
    import pub_dialogue.utils as du
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("dialogue_utils imported successfully")
except ImportError as _e:
    print(f"dialogue_utils not found: {_e}")
    print("Fetch it with: !wget https://raw.githubusercontent.com/mlatcl/pub-dialogue/main/dialogue_utils.py")
    raise

In [ ]:
# @title Configure API access
import os as _os

api_key = None

# 1. Try Colab secrets (when running in Google Colab)
try:
    from google.colab import userdata as _userdata
    api_key = _userdata.get('OPENAI_API_KEY')
    if api_key:
        print("API key loaded from Colab secrets")
except Exception:
    pass

# 2. Try environment variable (when running locally)
if not api_key:
    api_key = _os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("API key loaded from OPENAI_API_KEY environment variable")

# 3. Interactive prompt fallback
if not api_key:
    from getpass import getpass as _getpass
    api_key = _getpass("Enter OpenAI API key: ")
    print("API key entered manually")

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API connection verified")
except Exception as e:
    print(f"API error: {e}")

---
# SECTION 1: Data ingestion and preprocessing
---

In [ ]:
# @title Upload source PDF documents
show_status("Preparing PDF upload...")

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

pdf_files = []

if IN_COLAB:
    print("Upload your PDF files:")
    uploaded = _colab_files.upload()
    for filename, content in uploaded.items():
        if filename.endswith('.pdf'):
            filepath = PDF_FOLDER / filename
            filepath.write_bytes(content)
            pdf_files.append(filepath)
else:
    # Local execution: scan DATA_FOLDER for existing PDFs
    DATA_FOLDER.mkdir(parents=True, exist_ok=True)
    pdf_files = sorted(DATA_FOLDER.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files found in {DATA_FOLDER.resolve()}. "
            "Place your corpus PDFs there before running this cell."
        )
    print(f"Found {len(pdf_files)} PDFs in {DATA_FOLDER}/")

show_complete(f"Loaded {len(pdf_files)} PDF files")

In [ ]:
# @title Upload document metadata
show_status("Upload metadata file...")

metadata_df = None

if IN_COLAB:
    print("Upload Excel file with columns: filename, technology, year")
    uploaded = _colab_files.upload()
    for fn, content in uploaded.items():
        if fn.endswith(('.xlsx', '.xls')):
            path = OUTPUT_FOLDER / fn
            path.write_bytes(content)
            metadata_df = pd.read_excel(path)
            break
    if metadata_df is None:
        raise ValueError("No Excel file uploaded!")
else:
    # Local execution: look for an xlsx in OUTPUT_FOLDER then repo root
    _candidates = list(DATA_FOLDER.glob("*.xlsx")) + list(Path(".").glob("*.xlsx"))
    if not _candidates:
        raise FileNotFoundError(
            "No metadata xlsx found. Place tech_metadata.xlsx in outputs/ or repo root."
        )
    metadata_df = pd.read_excel(_candidates[0])
    print(f"Loaded metadata from {_candidates[0]}")

required = ['filename', 'technology', 'year']
missing = [c for c in required if c not in metadata_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

metadata_lookup = metadata_df.set_index('filename').to_dict('index')

print(f"\nTechnology Distribution:")
print(metadata_df['technology'].value_counts().to_string())
print(f"\nYear Range: {metadata_df['year'].min()} - {metadata_df['year'].max()}")

show_complete(f"Metadata loaded for {len(metadata_df)} documents")

In [ ]:
# @title Extract paragraph-level text from PDFs (v19 three-case hybrid chunker)
# Helper functions (_split_into_sentences, _repack_sentences_into_chunks,
# _paragraph_split, extract_chunks_from_pdf) are provided by dialogue_utils.
# The three-case strategy:
#   Case 1 — paragraph-only (most documents)
#   Case 2 — paragraphs + internal sentence-split of oversized paragraphs
#   Case 3 — full sentence-fallback (PDFs lacking paragraph breaks)

show_status("Extracting paragraph chunks...")

reset_chunk_stats()

all_chunks = []
for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    if pdf_path.name not in metadata_lookup:
        raise ValueError(f"No metadata found for PDF: {pdf_path.name}")
    metadata = metadata_lookup[pdf_path.name]
    chunks = extract_chunks_from_pdf(pdf_path, metadata)
    all_chunks.extend(chunks)

chunks_df = pd.DataFrame(all_chunks)
chunks_df["chunk_id"] = [f"chunk_{i}" for i in range(len(chunks_df))]

_chunk_stats = get_chunk_stats()

print(f"\nExtraction Summary:")
print(f"  Documents:                                     {chunks_df['source_file'].nunique()}")
print(f"  Documents w/ pure paragraph segmentation:      {_chunk_stats['documents_paragraph_only']}")
print(f"  Documents w/ paragraphs + some sentence-split: {_chunk_stats['documents_paragraph_with_split']}")
print(f"  Documents w/ full sentence-fallback:           {_chunk_stats['documents_sentence_fallback']}")
print(f"  Oversized paragraphs split internally:         {_chunk_stats['oversized_paragraphs_split']}")
print(f"  Chunks produced by sentence-split:             {_chunk_stats['chunks_from_sentence_split']}")
print(f"  Chunks produced by full sentence-fallback:     {_chunk_stats['chunks_from_sentence_fallback']}")
print(f"  Total chunks (paragraph-derived):              {(chunks_df['chunking_method']=='paragraph').sum()}")
print(f"  Paragraphs seen (raw):                         {_chunk_stats['paragraphs_seen']}")
print(f"  Below {MIN_CHUNK_WORDS}-word floor:                          {_chunk_stats['paragraphs_below_word_floor']}")
print(f"  Below {MIN_CHUNK_CHARS}-char floor:                          {_chunk_stats['paragraphs_below_char_floor']}")
print(f"  Truncated (safety net):                        {_chunk_stats['paragraphs_truncated']}")
print(f"  Kept after filter:                             {_chunk_stats['paragraphs_kept']}")
print(f"  Words per kept chunk (mean):                   {chunks_df['word_count'].mean():.0f}")

# Per-document summary
doc_summary = (chunks_df.groupby("source_file")
               .agg(paragraphs_kept=("chunk_id", "size"),
                    methods_used=("chunking_method", lambda s: "+".join(sorted(s.unique()))),
                    truncated_chunks=("was_truncated", "sum"))
               .reset_index()
               .sort_values("paragraphs_kept"))
doc_summary.to_csv(OUTPUT_FOLDER / "paragraph_chunks_per_document.csv", index=False)
print(f"\nDocuments using full sentence-fallback (PDF lacks paragraph breaks):")
fallback_docs = doc_summary[doc_summary["methods_used"] == "sentence_fallback"]
print(fallback_docs.to_string(index=False) if len(fallback_docs) else "  (none)")

chunks_df.to_csv(OUTPUT_FOLDER / "paragraph_chunks.csv", index=False)
show_complete(f"Extracted {len(chunks_df)} chunks from {chunks_df['source_file'].nunique()} documents")

TECHNOLOGY_CATEGORIES = sorted(metadata_df["technology"].dropna().unique().tolist())

# Sanity check: technology labels remain canonical
observed = set(chunks_df["technology_meta"].unique().tolist())
expected = set(TECHNOLOGY_CATEGORIES)
if observed != expected:
    print("\nWARNING: Technology labels in extracted chunks differ from metadata!")
    print("Observed-only:", sorted(observed - expected))
    print("Metadata-only:", sorted(expected - observed))

In [ ]:
# @title Summarise paragraph-level data quality

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Chunks by technology
tech_counts = chunks_df['technology_meta'].value_counts()
axes[0, 0].barh(tech_counts.index, tech_counts.values, color='steelblue')
axes[0, 0].set_xlabel('Number of Paragraphs')
axes[0, 0].set_title('Paragraphs by Technology')
axes[0, 0].invert_yaxis()

# Chunks by year
year_counts = chunks_df.groupby('year').size()
axes[0, 1].bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Paragraphs')
axes[0, 1].set_title('Paragraphs by Year')
axes[0, 1].tick_params(axis='x', rotation=45)

# Word count distribution
axes[1, 0].hist(chunks_df['word_count'], bins=30, color='steelblue', edgecolor='white')
axes[1, 0].set_xlabel('Word Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Paragraph Length Distribution')

# Chunks per document
doc_chunks = chunks_df.groupby('source_file').size().sort_values(ascending=False)
axes[1, 1].bar(range(len(doc_chunks)), doc_chunks.values, color='steelblue')
axes[1, 1].set_xlabel('Document Index')
axes[1, 1].set_ylabel('Paragraphs')
axes[1, 1].set_title('Paragraphs per Document')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "data_quality_overview.png", dpi=150)
plt.show()

In [ ]:
# @title (v16) Chunk content-quality diagnostic
# Reports the proportion of kept chunks that look like bibliography fragments
# or survey-table rows, so the analyst can see the contamination rate at the
# chosen MIN_CHUNK_WORDS floor.

show_status("Running chunk content-quality diagnostic...")

import re

# Heuristics. Kept deliberately simple — these are diagnostics, not filters.
_BIB_PATTERN = re.compile(r"\b(et al\.?|doi:|http(s)?://|pp?\.\s*\d+|vol\.|isbn|issn)\b", re.IGNORECASE)
_TABLE_PATTERN = re.compile(r"%")  # any percent-sign occurrence
_CITATION_YEAR = re.compile(r"\([12][0-9]{3}\)")  # (1999) / (2024) etc.

def _looks_like_bibliography(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return bool(_BIB_PATTERN.search(text)) and bool(_CITATION_YEAR.search(text))

def _looks_like_table_row(text: str) -> bool:
    if not isinstance(text, str):
        return False
    pct_count = len(_TABLE_PATTERN.findall(text))
    # Heuristic: 2+ percent signs in a chunk strongly suggests a survey table row
    return pct_count >= 2

chunks_df["likely_bibliography"] = chunks_df["text"].apply(_looks_like_bibliography)
chunks_df["likely_table_row"] = chunks_df["text"].apply(_looks_like_table_row)

n_total = len(chunks_df)
n_bib = int(chunks_df["likely_bibliography"].sum())
n_table = int(chunks_df["likely_table_row"].sum())

print(f"Chunk content quality (n = {n_total}):")
print(f"  Likely bibliography fragment: {n_bib}  ({n_bib/n_total*100:.1f}%)")
print(f"  Likely survey-table row:      {n_table}  ({n_table/n_total*100:.1f}%)")
print()
print("These chunks are kept for analysis (the diagnostic does not filter).")
print("If the contamination rate is high, consider raising MIN_CHUNK_WORDS.")
print(f"Current floor: {MIN_CHUNK_WORDS} words / {MIN_CHUNK_CHARS} chars.")

# Write the flagged chunks for manual inspection
flagged = chunks_df[chunks_df["likely_bibliography"] | chunks_df["likely_table_row"]][
    ["chunk_id", "source_file", "word_count", "likely_bibliography", "likely_table_row", "text"]
]
flagged.to_csv(OUTPUT_FOLDER / "chunk_quality_flagged.csv", index=False)
show_complete(f"Wrote {len(flagged)} flagged chunks to chunk_quality_flagged.csv (for inspection)")


---
# SECTION 2: Concern extraction
---

This section extracts decontextualised public concern phrases from paragraph-level dialogue text. These concern phrases form the primary analytic units for all subsequent embedding, clustering, and comparative analysis.


In [ ]:
# @title Extract decontextualised concern phrases

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

show_status("Extracting decontextualised concern phrases...")

EXTRACTION_PROMPT = """Extract the core public concerns from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.)
2. Extract the underlying concern that could apply to ANY technology
3. Keep phrases concise (3-10 words each)
4. Focus on what people are worried about, not factual statements

EXAMPLES:
- "People worried about AI making unfair decisions" → "unfair automated decisions"
- "Concerns about nuclear waste storage" → "long-term waste storage safety"
- "Distrust of government handling of genetic data" → "distrust of government data handling"
- "Fear that robots will take jobs" → "technology displacing employment"

Return 1-3 concern phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public concern, return "NO_CONCERN".

Paragraph:
{text}"""

_concern_extraction_failures = []  # list of (chunk_id, reason)
_concern_extraction_failures_lock = threading.Lock()

def _record_failure(chunk_id, reason):
    with _concern_extraction_failures_lock:
        _concern_extraction_failures.append((chunk_id, reason))

_TECH_TERM_PATTERN = re.compile(
    r"\b("
    r"ai|artificial intelligence|"
    r"nuclear|"
    r"genetic|genome|gene|crispr|embryo|stem cell|"
    r"nano|"
    r"robot|drone|autonomous|"
    r"quantum|"
    r"gm|"
    r"algorithm|algorithmic|automated|automation|"
    r"radiation|engineered|biometric|surveillance"
    r")\b",
    re.IGNORECASE,
)

def _phrase_contains_tech_term(phrase: str) -> bool:
    return bool(_TECH_TERM_PATTERN.search(phrase or ""))

def extract_concerns_from_paragraph(row_tuple):
    """Extract decontextualised concerns from a single paragraph.

    temperature=1). Logs failures instead of silently returning [].
    'max_tokens reached' BadRequestError on ~0.7% of chunks). One retry on
    transient API errors. Word-boundary regex for the post-extraction
    technology-term filter (was substring match).
    """
    idx, row = row_tuple
    chunk_id = row['chunk_id']

    def _call():
        return client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Extract public concerns. Be concise. Remove technology-specific language."},
                {"role": "user", "content": EXTRACTION_PROMPT.format(text=row['text'][:2000])}
            ],
            max_completion_tokens=1500,
        )

    try:
        try:
            response = _call()
        except Exception:
            time.sleep(1.0)
            response = _call()  # one retry
        content = response.choices[0].message.content
        if content is None:
            _record_failure(chunk_id, "empty_response")
            return chunk_id, []
        content = content.strip()

        if content == "NO_CONCERN" or not content:
            return chunk_id, []

        concerns = [c.strip() for c in content.split('\n') if c.strip() and c.strip() != "NO_CONCERN"]
        filtered = [c for c in concerns if not _phrase_contains_tech_term(c)]
        return chunk_id, filtered
    except Exception as e:
        _record_failure(chunk_id, f"exception: {type(e).__name__}: {e}")
        return chunk_id, []

# Check for cached extractions
concerns_cache_file = CHECKPOINT_FOLDER / "extracted_concerns.json"

if concerns_cache_file.exists():
    print("Found cached concern extractions...")
    with open(concerns_cache_file) as f:
        cached_concerns = json.load(f)

    cached_ids = set(cached_concerns.keys())
    current_ids = set(chunks_df['chunk_id'].tolist())

    if cached_ids == current_ids:
        print(f"Using cached extractions for {len(cached_concerns)} paragraphs")
        all_concerns = cached_concerns
    else:
        print("Cache mismatch - will re-extract")
        all_concerns = None
else:
    all_concerns = None

if all_concerns is None:
    all_concerns = {}

    # Build lookup for metadata
    chunk_metadata = chunks_df.set_index('chunk_id')[['technology', 'year', 'source_file']].to_dict('index')

    # Parallel extraction with 10 workers
    MAX_WORKERS = 10
    rows = list(chunks_df.iterrows())

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_concerns_from_paragraph, row): row[1]['chunk_id']
                   for row in rows}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting concerns"):
            chunk_id, concerns = future.result()
            meta = chunk_metadata[chunk_id]
            all_concerns[chunk_id] = {
                'concerns': concerns,
                'technology': meta['technology'],
                'year': int(meta['year']) if pd.notna(meta['year']) else None,
                'source_file': meta['source_file']
            }

    # Cache the results
    with open(concerns_cache_file, 'w') as f:
        json.dump(all_concerns, f, indent=2)

    show_complete(f"Extracted concerns from {len(all_concerns)} paragraphs")

if _concern_extraction_failures:
    failure_log_path = OUTPUT_FOLDER / "concern_extraction_failures.csv"
    pd.DataFrame(_concern_extraction_failures, columns=["chunk_id", "reason"]).to_csv(failure_log_path, index=False)
    failure_rate = len(_concern_extraction_failures) / max(1, len(chunks_df))
    show_warning(
        f"Concern extraction had {len(_concern_extraction_failures)} failures "
        f"({failure_rate:.1%} of chunks). See {failure_log_path}."
    )
    if failure_rate > 0.05:
        raise RuntimeError(
            f"Concern extraction failure rate ({failure_rate:.1%}) exceeds 5% threshold. "
            f"Inspect {failure_log_path} before proceeding."
        )

# Flatten to individual concern rows
concern_rows = []
for chunk_id, data in all_concerns.items():
    for concern in data['concerns']:
        concern_rows.append({
            'chunk_id': chunk_id,
            'concern': concern,
            'technology': data['technology'],
            'year': data['year'],
            'source_file': data['source_file']
        })

concerns_df = pd.DataFrame(concern_rows)
concerns_df['concern_id'] = [f"concern_{i}" for i in range(len(concerns_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs processed: {len(all_concerns)}")
print(f"  Total concern phrases: {len(concerns_df)}")
print(f"  Avg concerns per paragraph: {len(concerns_df) / max(1, len(all_concerns)):.2f}")
print(f"  Paragraphs with no concerns: {sum(1 for d in all_concerns.values() if len(d['concerns']) == 0)}")

concerns_df.to_csv(OUTPUT_FOLDER / "extracted_concerns.csv", index=False)


In [ ]:
# Make sure concerns_df knows the technology of each paragraph
concerns_df = concerns_df.merge(
    chunks_df[["chunk_id", "technology_meta"]],
    on="chunk_id",
    how="left"
)


In [ ]:
# @title Inspect sample concern extractions

print("Sample extracted concerns by technology:")
print("(Checking that technology-specific language has been removed)\n")

for tech in concerns_df['technology_meta'].unique()[:6]:
    tech_concerns = concerns_df[concerns_df['technology_meta'] == tech]['concern'].head(8).tolist()
    print(f"\n{tech}:")
    for c in tech_concerns:
        print(f"  • {c}")

---
# SECTION 3: Embedding and clustering
---

In [ ]:
# @title Generate semantic embeddings for concern phrases

def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in response.data])

embeddings_file = CHECKPOINT_FOLDER / "concern_embeddings.npy"
concern_ids_file = CHECKPOINT_FOLDER / "concern_ids.json"

if embeddings_file.exists() and concern_ids_file.exists():
    print("Found cached embeddings...")
    embeddings = np.load(embeddings_file)
    with open(concern_ids_file) as f:
        cached_concern_ids = json.load(f)

    if cached_concern_ids == concerns_df['concern_id'].tolist():
        print(f"Using cached embeddings: {embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    show_status(f"Generating embeddings for {len(concerns_df)} concern phrases...")

    texts = concerns_df['concern'].tolist()
    all_embeddings = []
    failed_batch_starts = []  # track failed batches

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i+EMBEDDING_BATCH_SIZE]
        try:
            batch_embeddings = get_embeddings_batch(batch)
            all_embeddings.append(batch_embeddings)
        except Exception as e:
            print(f"Error on batch {i}: {e}")
            failed_batch_starts.append(i)
        time.sleep(0.1)

    if failed_batch_starts:
        raise RuntimeError(
            f"{len(failed_batch_starts)} embedding batch(es) failed: "
            f"starting indices {failed_batch_starts}. "
            f"Re-run this cell after investigating; the cache will be regenerated."
        )

    embeddings = np.vstack(all_embeddings)
    np.save(embeddings_file, embeddings)
    with open(concern_ids_file, 'w') as f:
        json.dump(concerns_df['concern_id'].tolist(), f)

    show_complete(f"Generated embeddings: {embeddings.shape}")

assert embeddings.shape[0] == len(concerns_df), (
    f"Embeddings count ({embeddings.shape[0]}) does not match concerns_df rows ({len(concerns_df)}). "
    f"Delete {embeddings_file} and re-run."
)

print(f"Embedding dimensions: {embeddings.shape[1]}")


---
# SECTION 3B: Embedding and clustering (benefits)
---

In [ ]:
# @title Extract decontextualised benefit phrases

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import json
import pandas as pd

show_status("Extracting decontextualised benefit phrases...")

BENEFIT_EXTRACTION_PROMPT = """Extract the core public BENEFITS (upsides, hoped-for gains, opportunities) from this paragraph from a UK public dialogue report.

CRITICAL RULES:
1. Remove ALL technology-specific references (AI, nuclear, genetic, nano, etc.).
2. Extract the underlying benefit that could apply to ANY emerging technology.
3. Keep each benefit phrase concise (3–10 words).
4. Prefer concrete impacts over vague praise (e.g., "faster diagnosis" not "innovation").
5. Do NOT include concerns, caveats, or neutral facts unless they clearly express a benefit.

EXAMPLES:
- "AI could help doctors spot cancers earlier" → "earlier disease detection"
- "Nuclear could provide reliable low-carbon energy" → "reliable low-carbon energy supply"
- "Genetic research might enable personalised treatments" → "more personalised medical treatment"
- "Robots could take on dangerous tasks" → "reduced human exposure to danger"
- "Better public services through improved efficiency" → "more efficient public services"

Return 1–3 benefit phrases, one per line. No bullets, no numbering.
If the paragraph contains no clear public benefit, return "NO_BENEFIT".

Paragraph:
{text}"""

_benefit_extraction_failures = []
_benefit_extraction_failures_lock = threading.Lock()

def _record_benefit_failure(chunk_id, reason):
    with _benefit_extraction_failures_lock:
        _benefit_extraction_failures.append((chunk_id, reason))

_BENEFIT_TECH_TERM_PATTERN = re.compile(
    r"\b("
    r"ai|artificial intelligence|"
    r"nuclear|"
    r"genetic|genome|gene|crispr|embryo|stem cell|"
    r"nano|"
    r"robot|drone|autonomous|"
    r"quantum|"
    r"gm|"
    r"algorithm|algorithmic|automated|automation|"
    r"radiation|engineered|biometric|surveillance|"
    r"neural|brain-computer"
    r")\b",
    re.IGNORECASE,
)

def _benefit_phrase_contains_tech_term(phrase: str) -> bool:
    return bool(_BENEFIT_TECH_TERM_PATTERN.search(phrase or ""))

def extract_benefits_from_paragraph(row_tuple):
    """Extract decontextualised benefits from a single paragraph.

    temperature=1). Logs failures.
    errors, word-boundary regex for the post-extraction technology-term filter.
    """
    idx, row = row_tuple
    chunk_id = row["chunk_id"]

    paragraph_text = str(row["text"])[:2000]

    def _call():
        return client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Extract public benefits. Be concise. Remove technology-specific language."},
                {"role": "user", "content": BENEFIT_EXTRACTION_PROMPT.format(text=paragraph_text)}
            ],
            max_completion_tokens=1500,
        )

    try:
        try:
            response = _call()
        except Exception:
            time.sleep(1.0)
            response = _call()  # one retry

        content = response.choices[0].message.content
        if content is None:
            _record_benefit_failure(chunk_id, "empty_response")
            return chunk_id, []

        content = content.strip()

        # Guard against model refusing due to missing text / page references
        if "I don't have the text" in content or "I don’t have the text" in content:
            _record_benefit_failure(chunk_id, "model_refused_no_text")
            return chunk_id, []

        if content == "NO_BENEFIT" or not content:
            return chunk_id, []

        benefits = [b.strip() for b in content.split("\n") if b.strip() and b.strip() != "NO_BENEFIT"]
        filtered = [b for b in benefits if not _benefit_phrase_contains_tech_term(b)]
        return chunk_id, filtered

    except Exception as e:
        _record_benefit_failure(chunk_id, f"exception: {type(e).__name__}: {e}")
        return chunk_id, []

# Cache file (benefits)
benefits_cache_file = CHECKPOINT_FOLDER / "extracted_benefits.json"

# Load cache if valid
if benefits_cache_file.exists():
    print("Found cached benefit extractions...")
    with open(benefits_cache_file) as f:
        cached_benefits = json.load(f)

    cached_ids = set(cached_benefits.keys())
    current_ids = set(chunks_df["chunk_id"].tolist())

    if cached_ids == current_ids:
        print(f"Using cached extractions for {len(cached_benefits)} paragraphs")
        all_benefits = cached_benefits
    else:
        print("Cache mismatch - will re-extract benefits")
        all_benefits = None
else:
    all_benefits = None

if all_benefits is None:
    all_benefits = {}

    # Metadata lookup
    chunk_metadata = chunks_df.set_index("chunk_id")[["technology", "year", "source_file"]].to_dict("index")

    MAX_WORKERS = 10
    rows = list(chunks_df.iterrows())

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_benefits_from_paragraph, row): row[1]["chunk_id"] for row in rows}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting benefits"):
            chunk_id, benefits = future.result()
            meta = chunk_metadata[chunk_id]
            all_benefits[chunk_id] = {
                "benefits": benefits,
                "technology": meta["technology"],
                "year": int(meta["year"]) if pd.notna(meta["year"]) else None,
                "source_file": meta["source_file"]
            }

    with open(benefits_cache_file, "w") as f:
        json.dump(all_benefits, f, indent=2)

    show_complete(f"Extracted benefits from {len(all_benefits)} paragraphs")

if _benefit_extraction_failures:
    failure_log_path = OUTPUT_FOLDER / "benefit_extraction_failures.csv"
    pd.DataFrame(_benefit_extraction_failures, columns=["chunk_id", "reason"]).to_csv(failure_log_path, index=False)
    failure_rate = len(_benefit_extraction_failures) / max(1, len(chunks_df))
    show_warning(
        f"Benefit extraction had {len(_benefit_extraction_failures)} failures "
        f"({failure_rate:.1%} of chunks). See {failure_log_path}."
    )
    if failure_rate > 0.05:
        raise RuntimeError(
            f"Benefit extraction failure rate ({failure_rate:.1%}) exceeds 5% threshold. "
            f"Inspect {failure_log_path} before proceeding."
        )

# Flatten to individual benefit rows
benefit_rows = []
for chunk_id, data in all_benefits.items():
    for benefit in data["benefits"]:
        benefit_rows.append({
            "chunk_id": chunk_id,
            "benefit": benefit,
            "technology": data["technology"],
            "year": data["year"],
            "source_file": data["source_file"]
        })

benefits_df = pd.DataFrame(benefit_rows)
benefits_df["benefit_id"] = [f"benefit_{i}" for i in range(len(benefits_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs processed: {len(all_benefits)}")
print(f"  Total benefit phrases: {len(benefits_df)}")
print(f"  Avg benefits per paragraph: {len(benefits_df) / max(1, len(all_benefits)):.2f}")
print(f"  Paragraphs with no benefits: {sum(1 for d in all_benefits.values() if len(d['benefits']) == 0)}")

benefits_df.to_csv(OUTPUT_FOLDER / "extracted_benefits.csv", index=False)


In [ ]:
# Make sure benefits_df knows the technology of each paragraph (safe merge)

# Prefer chunks_df's 'technology' if 'technology_meta' doesn't exist there
tech_col = "technology_meta" if "technology_meta" in chunks_df.columns else "technology"

benefits_df = benefits_df.merge(
    chunks_df[["chunk_id", tech_col]].rename(columns={tech_col: "technology_meta"}),
    on="chunk_id",
    how="left"
)

# If benefits_df already has 'technology' and it's missing where technology_meta exists, fill it
if "technology" in benefits_df.columns:
    benefits_df["technology"] = benefits_df["technology"].fillna(benefits_df["technology_meta"])
else:
    benefits_df["technology"] = benefits_df["technology_meta"]

In [ ]:
# @title Inspect sample benefit extractions

import pandas as pd

print("Sample extracted benefits by technology:")
print("(Checking that technology-specific language has been removed)\n")

# Identify the text column in benefits_df
candidate_cols = ["benefit", "benefit_phrase", "phrase", "extracted_phrase", "text"]
phrase_col = next((c for c in candidate_cols if c in benefits_df.columns), None)

if phrase_col is None:
    raise KeyError(
        f"Couldn't find a benefit text column in benefits_df. "
        f"Looked for {candidate_cols}. Available columns: {list(benefits_df.columns)}"
    )

# Identify technology column (you appear to have technology_meta)
tech_col = "technology_meta" if "technology_meta" in benefits_df.columns else (
    "technology" if "technology" in benefits_df.columns else None
)

if tech_col is None:
    raise KeyError(
        f"Couldn't find a technology column in benefits_df. "
        f"Available columns: {list(benefits_df.columns)}"
    )

for tech in benefits_df[tech_col].dropna().unique()[:6]:
    tech_benefits = benefits_df[benefits_df[tech_col] == tech][phrase_col].head(8).tolist()
    print(f"\n{tech}:")
    for b in tech_benefits:
        print(f"  • {b}")

In [ ]:
# @title Generate semantic embeddings for benefit phrases

# Reuse get_embeddings_batch if already defined in Part I
if "get_embeddings_batch" not in globals():
    def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
        response = client.embeddings.create(input=texts, model=model)
        return np.array([item.embedding for item in response.data])

embeddings_file = CHECKPOINT_FOLDER / "benefit_embeddings.npy"
benefit_ids_file = CHECKPOINT_FOLDER / "benefit_ids.json"

if embeddings_file.exists() and benefit_ids_file.exists():
    print("Found cached embeddings...")
    benefit_embeddings = np.load(embeddings_file)
    with open(benefit_ids_file) as f:
        cached_benefit_ids = json.load(f)

    if cached_benefit_ids == benefits_df["benefit_id"].tolist():
        print(f"Using cached embeddings: {benefit_embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        benefit_embeddings = None
else:
    benefit_embeddings = None

if benefit_embeddings is None:
    show_status(f"Generating embeddings for {len(benefits_df)} benefit phrases...")

    texts = benefits_df["benefit"].tolist()
    all_embeddings = []
    failed_batch_starts = []

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i+EMBEDDING_BATCH_SIZE]
        try:
            batch_embeddings = get_embeddings_batch(batch)
            all_embeddings.append(batch_embeddings)
        except Exception as e:
            print(f"Error on batch {i}: {e}")
            failed_batch_starts.append(i)
        time.sleep(0.1)

    if failed_batch_starts:
        raise RuntimeError(
            f"{len(failed_batch_starts)} benefit embedding batch(es) failed: "
            f"starting indices {failed_batch_starts}. "
            f"Re-run this cell after investigating; the cache will be regenerated."
        )

    benefit_embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 0))
    np.save(embeddings_file, benefit_embeddings)
    with open(benefit_ids_file, "w") as f:
        json.dump(benefits_df["benefit_id"].tolist(), f)

    show_complete(f"Generated embeddings: {benefit_embeddings.shape}")

assert benefit_embeddings.shape[0] == len(benefits_df), (
    f"Benefit embeddings count ({benefit_embeddings.shape[0]}) does not match "
    f"benefits_df rows ({len(benefits_df)}). Delete {embeddings_file} and re-run."
)

if benefit_embeddings.size:
    print(f"Embedding dimensions: {benefit_embeddings.shape[1]}")
else:
    print("No embeddings generated (benefits_df empty).")


In [ ]:
# @title Extraction checkpoint — confirm artifacts are saved
# Run this at the end of 01_processing to verify that all raw artifacts
# have been written before starting 01a_clustering.
from pathlib import Path as _Path
_out  = OUTPUT_FOLDER
_ckpt = CHECKPOINT_FOLDER

_EXPECTED = {
    _out  / "paragraph_chunks.csv":     "chunks",
    _out  / "extracted_concerns.csv":   "concern phrases",
    _out  / "extracted_benefits.csv":   "benefit phrases",
    _ckpt / "concern_embeddings.npy":   "concern embeddings",
    _ckpt / "benefit_embeddings.npy":   "benefit embeddings",
}

_ok = True
for _p, _label in _EXPECTED.items():
    _sz = f"{_p.stat().st_size / 1e6:.1f} MB" if _p.exists() else "MISSING"
    _flag = "OK  " if _p.exists() else "MISS"
    print(f"  {_flag}  {_label:<25}  {_sz}")
    if not _p.exists():
        _ok = False

if not _ok:
    raise RuntimeError("Some extraction artifacts are missing — check earlier cells.")
print("\nAll extraction artifacts present. Run 01a_clustering.ipynb next.")